## ⚙️ CONFIG — Тохиргоо

In [ ]:
import pandas as pd
import gc

# =========================================================
# CONFIG
# =========================================================
RISK_FILE   = 'Тооцох харилцагч to DAA.xlsx'
START_DATE  = '2025-01-01'
MIN_AMOUNT  = 20_000_000   # 20 сая+
MIN_COUNT   = 10            # CIF-д нийт 10+ гүйлгээ
MIN_DEST    = 5             # 5+ өөр данс руу гүйлгээ хийсэн байх
OUTPUT_FILE = 'Шаардлага_10plus_20M_5dest_Debit.xlsx'
ACID_BATCH  = 1000          # batch хэмжээ (500 → 1000)

# =========================================================
# STEP 0: CUSTOMERID баганаас CIF унших
# =========================================================
df_risk = pd.read_excel(RISK_FILE, dtype=str)
cid_col = next((c for c in df_risk.columns if c.strip().upper() == 'CUSTOMERID'), None)
if cid_col is None:
    raise ValueError(f"'CUSTOMERID' багана олдсонгүй. Баганауд: {list(df_risk.columns)}")

RISK_CIFS = set(
    df_risk[cid_col].dropna().astype(str).str.strip()
    .str.replace(r'\.0$', '', regex=True)
)
print(f"✅ Шүүх CIF: {len(RISK_CIFS):,}")


## 🏦 АЛХАМ 1 — CIF-тэй дансдыг татах

In [ ]:
print("⏳ АЛХАМ 1: Дансны мэдээлэл татаж байна...")

acc_parts     = []
cif_list      = list(RISK_CIFS)
total_batches = -(-len(cif_list) // ACID_BATCH)

for i, start in enumerate(range(0, len(cif_list), ACID_BATCH), 1):
    batch   = cif_list[start : start + ACID_BATCH]
    cifs_in = "('" + "','".join(batch) + "')"
    print(f"  Batch {i}/{total_batches}...", end="\r")
    try:
        df_a = client.query_df(f"""
            SELECT G_ACID AS ACID, G_FORACID AS FORACID,
                   toString(G_CIF_ID) AS CIF_ID,
                   toDate(G_ACCT_OPN_DATE) AS ACCT_OPN_DATE,
                   G_SCHM_CODE AS SCHM_CODE, 'TAM' AS SRC
            FROM FINACLE.GAM_TAM
            WHERE toString(G_CIF_ID) IN {cifs_in}
            UNION ALL
            SELECT G_ACID AS ACID, G_FORACID AS FORACID,
                   toString(G_CIF_ID) AS CIF_ID,
                   toDate(G_ACCT_OPN_DATE) AS ACCT_OPN_DATE,
                   G_SCHM_CODE AS SCHM_CODE, 'SMT' AS SRC
            FROM FINACLE.GAM_SMT
            WHERE toString(G_CIF_ID) IN {cifs_in}
        """)
        if not df_a.empty:
            acc_parts.append(df_a)
    except Exception as e:
        print(f"\n  ⚠️ Batch {i}: {e}")

df_accounts = pd.concat(acc_parts, ignore_index=True).drop_duplicates('ACID')
acid_list   = df_accounts['ACID'].astype(str).tolist()
print(f"\n✅ {len(df_accounts):,} данс олдлоо.")


## 🔢 АЛХАМ 2 — Гүйлгээний тоо (COUNT pre-filter)
> Join-гүй тул хурдан. MIN_COUNT хангаагүй CIF-уудыг таслан, дараа нь зөвхөн тэнцсэн ACIDуудын дэлгэрэнгүй татна.

In [ ]:
print(f"⏳ АЛХАМ 2: {len(acid_list):,} дансны гүйлгээний тоо шүүж байна...")

cnt_parts     = []
total_batches = -(-len(acid_list) // ACID_BATCH)

for i, start in enumerate(range(0, len(acid_list), ACID_BATCH), 1):
    batch    = acid_list[start : start + ACID_BATCH]
    acids_in = "('" + "','".join(batch) + "')"
    print(f"  Batch {i}/{total_batches}...", end="\r")
    try:
        df_c = client.query_df(f"""
            SELECT toString(H_ACID) AS ACID, count() AS TXN_COUNT
            FROM FINACLE.HTD_ATD
            PREWHERE H_TRAN_DATE >= toDate('{START_DATE}')
            WHERE H_DEL_FLG != 'Y'
              AND H_PART_TRAN_TYPE = 'D'
              AND H_TRAN_AMT >= {MIN_AMOUNT}
              AND H_ACID IN {acids_in}
            GROUP BY H_ACID
            SETTINGS max_execution_time = 120
        """)
        if not df_c.empty:
            cnt_parts.append(df_c)
    except Exception as e:
        print(f"\n  ⚠️ Batch {i}: {e}")

df_counts = (pd.concat(cnt_parts, ignore_index=True)
             if cnt_parts
             else pd.DataFrame(columns=['ACID', 'TXN_COUNT']))

# ACID → CIF холбож, CIF-д нийлүүлэн тоол
df_counts  = df_counts.merge(df_accounts[['ACID', 'CIF_ID']], on='ACID', how='left')
cif_totals = df_counts.groupby('CIF_ID')['TXN_COUNT'].sum()
pre_cifs   = cif_totals[cif_totals >= MIN_COUNT].index
pre_acids  = df_accounts[df_accounts['CIF_ID'].isin(pre_cifs)]['ACID'].astype(str).tolist()

print(f"\n✅ Pre-filter: {len(pre_cifs):,} CIF ({len(pre_acids):,} данс) MIN_COUNT хангав.")
print(f"   Хасагдсан:  {len(acid_list) - len(pre_acids):,} данс цаашид татахгүй → цаг хэмнэнэ.")


## 💳 АЛХАМ 3 — Дэлгэрэнгүй гүйлгээ + зуучийн данс (CREDIT_ACID)
> HTD_ATD self-join: ижил `H_TRAN_ID`-тэй debit ↔ credit талыг холбоно → хүлээн авагчийн данс гарна.

In [ ]:
print(f"⏳ АЛХАМ 3: {len(pre_acids):,} дансны дэлгэрэнгүй гүйлгээ татаж байна...")

txn_parts     = []
total_batches = -(-len(pre_acids) // ACID_BATCH)

for i, start in enumerate(range(0, len(pre_acids), ACID_BATCH), 1):
    batch    = pre_acids[start : start + ACID_BATCH]
    acids_in = "('" + "','".join(batch) + "')"
    print(f"  Batch {i}/{total_batches}...", end="\r")
    try:
        df_t = client.query_df(f"""
            SELECT
                toString(d.H_ACID)        AS ACID,
                toString(d.H_TRAN_ID)     AS TRAN_ID,
                toDate(d.H_TRAN_DATE)     AS TRAN_DATE,
                d.H_TRAN_AMT              AS DEBIT_AMT,
                d.H_SOL_ID                AS SOL_ID,
                d.H_TRAN_PARTICULAR       AS PARTICULARS,
                toString(c.H_ACID)        AS CREDIT_ACID,
                c.H_FORACID               AS CREDIT_FORACID
            FROM FINACLE.HTD_ATD d
            LEFT JOIN FINACLE.HTD_ATD c
                ON  d.H_TRAN_ID        = c.H_TRAN_ID
                AND c.H_PART_TRAN_TYPE = 'C'
                AND c.H_DEL_FLG        != 'Y'
            PREWHERE d.H_TRAN_DATE >= toDate('{START_DATE}')
            WHERE d.H_DEL_FLG        != 'Y'
              AND d.H_PART_TRAN_TYPE  = 'D'
              AND d.H_TRAN_AMT        >= {MIN_AMOUNT}
              AND d.H_ACID            IN {acids_in}
            ORDER BY d.H_ACID, d.H_TRAN_DATE
            SETTINGS max_execution_time = 300
        """)
        if not df_t.empty:
            txn_parts.append(df_t)
    except Exception as e:
        print(f"\n  ⚠️ Batch {i}: {e}")
    gc.collect()

if txn_parts:
    df_txns = pd.concat(txn_parts, ignore_index=True)
    df_txns = df_txns.merge(df_accounts, on='ACID', how='left')
    print(f"\n✅ {len(df_txns):,} мөр татагдлаа.")
else:
    print("❌ Гүйлгээ олдсонгүй.")


## 📊 АЛХАМ 4-6 — Шүүлт → Нэр → Тайлан

| Шүүлт | Тайлбар |
|---|---|
| `MIN_COUNT = 10` | CIF-д нийт 20M+ зарлага ≥ 10 удаа |
| `MIN_DEST  = 5`  | Тэр гүйлгээнүүд **5+ өөр данс** руу явсан байх |

In [ ]:
if not txn_parts:
    print("❌ Гүйлгээ байхгүй тул тайлан гаргахгүй.")
else:
    # =========================================================
    # АЛХАМ 4: CIF-д хоёр шүүлт нэгэн зэрэг хэрэглэх
    #   — MIN_COUNT (10): 20M+ зарлагын тоо >= 10
    #   — MIN_DEST  (5):  5+ өөр (credit) данс руу гүйлгээ
    # =========================================================
    cif_stats = (
        df_txns.groupby('CIF_ID')
        .agg(
            TXN_COUNT  = ('TRAN_ID',     'count'),
            DEST_COUNT = ('CREDIT_ACID', 'nunique')
        )
        .reset_index()
    )

    valid_cifs = cif_stats[
        (cif_stats['TXN_COUNT']  >= MIN_COUNT) &
        (cif_stats['DEST_COUNT'] >= MIN_DEST)
    ]['CIF_ID'].tolist()

    print("📊 Шүүлтийн үр дүн:")
    print(f"   COUNT >= {MIN_COUNT}           : {len(cif_stats[cif_stats['TXN_COUNT']  >= MIN_COUNT]):,} CIF")
    print(f"   COUNT >= {MIN_COUNT} + DEST >= {MIN_DEST}: {len(valid_cifs):,} CIF  ← эцсийн тооцоо")

    df_txns = df_txns[df_txns['CIF_ID'].isin(valid_cifs)].copy()

    # =========================================================
    # АЛХАМ 5: Харилцагчийн нэр (ACCOUNTS) + GSP татах
    # =========================================================
    cif_found     = df_txns['CIF_ID'].unique().tolist()
    name_parts    = []
    total_batches = -(-len(cif_found) // ACID_BATCH)
    print(f"\n⏳ Харилцагчийн нэр татаж байна ({len(cif_found):,} CIF)...")

    for i, start in enumerate(range(0, len(cif_found), ACID_BATCH), 1):
        batch   = cif_found[start : start + ACID_BATCH]
        cifs_in = "('" + "','".join(batch) + "')"
        print(f"  Batch {i}/{total_batches}...", end="\r")
        try:
            df_n = client.query_df(f"""
                SELECT toString(ORGKEY) AS CIF_ID,
                       any(CUST_LAST_NAME)  AS LAST_NAME,
                       any(CUST_FIRST_NAME) AS FIRST_NAME
                FROM FINACLE.ACCOUNTS
                WHERE toString(ORGKEY) IN {cifs_in}
                GROUP BY ORGKEY
            """)
            if not df_n.empty:
                name_parts.append(df_n)
        except Exception as e:
            print(f"\n  ⚠️ Batch {i}: {e}")

    df_names = (pd.concat(name_parts, ignore_index=True)
                if name_parts
                else pd.DataFrame(columns=['CIF_ID', 'LAST_NAME', 'FIRST_NAME']))
    df_txns  = df_txns.merge(df_names, on='CIF_ID', how='left')

    # GSP (product description)
    df_gsp  = client.query_df(
        "SELECT SCHM_CODE, any(SCHM_DESC) AS SCHM_DESC FROM FINACLE.GSP GROUP BY SCHM_CODE"
    )
    gsp_map = dict(zip(df_gsp['SCHM_CODE'], df_gsp['SCHM_DESC']))
    df_txns['SCHM_DESC'] = df_txns['SCHM_CODE'].map(gsp_map)

    # =========================================================
    # АЛХАМ 6: CIF-д гүйлгээ дугаарлах + тайлан гаргах
    # =========================================================
    df_txns = df_txns.sort_values(['CIF_ID', 'TRAN_DATE', 'TRAN_ID']).reset_index(drop=True)
    df_txns['GUILGEE_DUGAAR'] = df_txns.groupby('CIF_ID').cumcount() + 1

    report = df_txns[[
        'CIF_ID', 'LAST_NAME', 'FIRST_NAME',
        'GUILGEE_DUGAAR',
        'FORACID', 'ACCT_OPN_DATE', 'SCHM_CODE', 'SCHM_DESC', 'SRC',
        'TRAN_DATE', 'TRAN_ID', 'DEBIT_AMT', 'SOL_ID', 'PARTICULARS',
        'CREDIT_FORACID'
    ]].copy()
    report.columns = [
        'CIF ID', 'Овог', 'Нэр',
        'Гүйлгээний дугаар (CIF-д)',
        'Дансны дугаар', 'Данс нээгдсэн огноо',
        'Бүтээгдэхүүний код', 'Бүтээгдэхүүний нэр', 'Төрөл',
        'Гүйлгээний огноо', 'Гүйлгээний ID',
        'Зарлагын дүн', 'Салбар', 'Тайлбар',
        'Хүлээн авагчийн данс'       # ← шинэ багана
    ]

    print(f"\n✅ Нийт {report['CIF ID'].nunique():,} CIF,  {len(report):,} гүйлгээ.")
    display(report.head(30))

    report.to_excel(OUTPUT_FILE, index=False)
    print(f"📊 '{OUTPUT_FILE}' хадгалагдлаа.")

    # Санах ой цэвэрлэх
    del txn_parts, acc_parts, cnt_parts, name_parts
    gc.collect()
